In [1]:
# ============================================================
# NHL PREDICTION MODEL - PRODUCTION VERSION
# ============================================================
# This is the clean, final version of the model
# No experiments or testing - just what works

# ============================================================
# PART 1: SETUP & DATA LOADING
# ============================================================

# Cell 1: Imports
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from datetime import datetime
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("NHL Prediction Model - Production Version")
print(f"Date: {datetime.now().strftime('%B %d, %Y')}\n")

NHL Prediction Model - Production Version
Date: March 19, 2026



In [2]:
# Cell 2: Database Connection
DB_HOST = "db.wluxnhuyfumkannjvgiu.supabase.co"
DB_PORT = "5432"
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASS = "BlueISland$$87"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"sslmode": "require"}
)

print("✅ Connected to Supabase\n")

✅ Connected to Supabase



In [3]:

# Cell 3: Load Game Data
query_games = """
    SELECT 
        "gameId",
        season,
        team,
        "opposingTeam",
        "goalsFor",
        "goalsAgainst",
        home_or_away,
        "gameDate",
        "xGoalsFor",
        "xGoalsAgainst",
        "corsiPercentage",
        "shotsOnGoalFor",
        "shotsOnGoalAgainst",
        "scoreAdjustedShotsAttemptsFor",
        "scoreAdjustedShotsAttemptsAgainst",
        "highDangerxGoalsFor",
        "highDangerxGoalsAgainst"
    FROM "NHL_Gamelog"
    WHERE situation = 'all'
    AND position = 'Team Level'
    AND "playoffGame" = '0'
    AND season IN (2024, 2025)
"""

games_df = pd.read_sql(query_games, engine)

# Cast numeric columns (stored as text in Supabase)
numeric_cols = [
    'goalsFor', 'goalsAgainst', 'xGoalsFor', 'xGoalsAgainst',
    'corsiPercentage', 'shotsOnGoalFor', 'shotsOnGoalAgainst',
    'scoreAdjustedShotsAttemptsFor', 'scoreAdjustedShotsAttemptsAgainst',
    'highDangerxGoalsFor', 'highDangerxGoalsAgainst'
]
games_df[numeric_cols] = games_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Split by season
prior_season = games_df[games_df['season'] == 2024].copy()
current_season = games_df[games_df['season'] == 2025].copy()

print(f"Prior season (2024-25): {len(prior_season['gameId'].unique())} games")
print(f"Current season (2025-26): {len(current_season['gameId'].unique())} games\n")

Prior season (2024-25): 1312 games
Current season (2025-26): 1082 games



In [4]:
# ============================================================
# PART 2: MODEL CLASSES
# ============================================================

# Cell 4: Bradley-Terry Model Class
class BradleyTerryModel:
    """
    Bradley-Terry model for rating teams based on win/loss results.
    Uses maximum likelihood to estimate team strengths.
    """
    
    def __init__(self):
        self.team_strengths = {}
        self.teams = []
        self.team_records = {}
    
    def fit(self, games_df):
        """Fit model on game results"""
        self.teams = sorted(list(set(games_df['team'])))
        n_teams = len(self.teams)
        team_to_idx = {team: i for i, team in enumerate(self.teams)}
        
        wins = np.zeros(n_teams)
        games_played = np.zeros((n_teams, n_teams))
        
        for team in self.teams:
            self.team_records[team] = {'W': 0, 'L': 0, 'T': 0}
        
        processed_games = set()
        
        for _, row in games_df.iterrows():
            game_id = row['gameId']
            if game_id in processed_games:
                continue
            processed_games.add(game_id)
            
            team = row['team']
            opponent = row['opposingTeam']
            goals_for = row['goalsFor']
            goals_against = row['goalsAgainst']
            
            team_idx = team_to_idx[team]
            opp_idx = team_to_idx[opponent]
            
            games_played[team_idx, opp_idx] += 1
            games_played[opp_idx, team_idx] += 1
            
            if goals_for > goals_against:
                wins[team_idx] += 1
                self.team_records[team]['W'] += 1
                self.team_records[opponent]['L'] += 1
            elif goals_against > goals_for:
                wins[opp_idx] += 1
                self.team_records[team]['L'] += 1
                self.team_records[opponent]['W'] += 1
            else:
                wins[team_idx] += 0.5
                wins[opp_idx] += 0.5
                self.team_records[team]['T'] += 1
                self.team_records[opponent]['T'] += 1
        
        # Maximum likelihood estimation
        def neg_log_likelihood(log_strengths):
            strengths = np.exp(log_strengths)
            ll = 0
            for i in range(n_teams):
                for j in range(n_teams):
                    if i != j and games_played[i, j] > 0:
                        p_ij = strengths[i] / (strengths[i] + strengths[j])
                        if wins[i] > 0:
                            expected_wins_ij = (games_played[i, j] / games_played[i, :].sum()) * wins[i]
                            ll += expected_wins_ij * np.log(p_ij + 1e-10)
                            ll += (games_played[i, j] - expected_wins_ij) * np.log(1 - p_ij + 1e-10)
            return -ll
        
        initial_strengths = np.zeros(n_teams)
        for i, team in enumerate(self.teams):
            team_games = games_played[i, :].sum()
            if team_games > 0:
                win_rate = max(0.01, min(0.99, wins[i] / team_games))
                initial_strengths[i] = np.log(win_rate / (1 - win_rate))
        
        result = minimize(neg_log_likelihood, initial_strengths, method='BFGS')
        
        raw_strengths = np.exp(result.x)
        raw_strengths = raw_strengths / np.mean(raw_strengths)
        
        self.team_strengths = {team: raw_strengths[i] for i, team in enumerate(self.teams)}
        
        return self
    
    def get_team_strength(self, team):
        """Get strength rating for a team"""
        if team not in self.team_strengths:
            raise ValueError(f"Team '{team}' not found")
        return self.team_strengths[team]

In [5]:
# Cell 5: Current Season Metrics Calculator
class CurrentSeasonMetrics:
    """Calculate team strengths from advanced metrics (xG, Corsi, etc.)"""
    
    def __init__(self, current_season_df):
        self.metric_strengths = {}
        if len(current_season_df) > 0:
            self._calculate(current_season_df)
    
    def _calculate(self, df):
        """Calculate metric-based strengths"""
        team_metrics = []
        
        for team in df['team'].unique():
            team_games = df[df['team'] == team]
            n = len(team_games)
            
            # Calculate per-game averages
            xg_diff = (team_games['xGoalsFor'].sum() - team_games['xGoalsAgainst'].sum()) / n
            corsi_pct = team_games['corsiPercentage'].mean() - 0.50
            shots_diff = (team_games['shotsOnGoalFor'].sum() - team_games['shotsOnGoalAgainst'].sum()) / n
            score_adj_diff = (team_games['scoreAdjustedShotsAttemptsFor'].sum() - 
                            team_games['scoreAdjustedShotsAttemptsAgainst'].sum()) / n
            hd_diff = (team_games['highDangerxGoalsFor'].sum() - team_games['highDangerxGoalsAgainst'].sum()) / n
            
            team_metrics.append({
                'team': team,
                'xg_diff': xg_diff,
                'corsi_pct': corsi_pct,
                'shots_diff': shots_diff,
                'score_adj_diff': score_adj_diff,
                'hd_diff': hd_diff
            })
        
        metrics_df = pd.DataFrame(team_metrics)
        
        # Normalize to z-scores
        normalized = pd.DataFrame({'team': metrics_df['team']})
        for metric in ['xg_diff', 'corsi_pct', 'shots_diff', 'score_adj_diff', 'hd_diff']:
            mean = metrics_df[metric].mean()
            std = metrics_df[metric].std()
            normalized[metric] = (metrics_df[metric] - mean) / std if std > 0 else 0
        
        # Weighted composite (same weights as validation)
        weights = {'xg_diff': 0.35, 'corsi_pct': 0.25, 'shots_diff': 0.20, 
                  'score_adj_diff': 0.10, 'hd_diff': 0.10}
        
        for _, row in normalized.iterrows():
            team = row['team']
            score = sum(weights[k] * row[k] for k in weights.keys())
            self.metric_strengths[team] = np.exp(score)
        
        # Normalize
        avg = np.mean(list(self.metric_strengths.values()))
        self.metric_strengths = {t: s/avg for t, s in self.metric_strengths.items()}
    
    def get_team_strength(self, team):
        """Get metric-based strength for a team"""
        return self.metric_strengths.get(team, 1.0)


In [6]:

# Cell 6: Production Model
class NHLPredictionModel:
    """
    Production NHL prediction model.
    
    Blends:
    - Prior season results (W/L)
    - Current season results (W/L) 
    - Current season metrics (xG, Corsi, etc.)
    - Goalie adjustments (GSAx)
    
    Weights adjust based on current season games played.
    """
    
    def __init__(self, prior_season_df, current_season_df, goalie_df=None):
        """Train the model"""
        print("Training model...")
        
        # Train prior season Bradley-Terry
        print("  - Prior season Bradley-Terry...")
        self.prior_bt = BradleyTerryModel().fit(prior_season_df)
        
        # Current season stats
        self.current_games = len(current_season_df['gameId'].unique())
        print(f"  - Current season: {self.current_games} games")
        
        # Train current season models
        if self.current_games >= 10:
            print("  - Current season Bradley-Terry...")
            self.current_bt = BradleyTerryModel().fit(current_season_df)
            self.has_bt = True
        else:
            self.has_bt = False
        
        print("  - Current season metrics...")
        self.current_metrics = CurrentSeasonMetrics(current_season_df)
        
        # Load goalie data if provided
        if goalie_df is not None:
            print("  - Loading goalie ratings...")
            self._load_goalies(goalie_df)
        else:
            self.goalie_ratings = {}
        
        # Calculate blending weights
        self._calculate_weights()
        
        # Blend all sources
        self._blend_strengths()
        
        print(f"\n✅ Model trained successfully!")
        print(f"   Weights: {self.prior_w:.0%} prior | {self.current_wl_w:.0%} current W/L | {self.current_met_w:.0%} metrics")
        if self.goalie_ratings:
            print(f"   Goalies loaded: {len(self.goalie_ratings)} goalies\n")
    
    def _load_goalies(self, goalie_df):
        """Load and rate goalies based on GSAx (Goals Saved Above Expected)"""
        # Filter for 'all' situation and minimum games played
        goalie_stats = goalie_df[
            (goalie_df['situation'] == 'all') & 
            (goalie_df['games_played'] >= 5)  # Min 5 games
        ].copy()
        
        # Calculate GSAx (Goals Saved Above Expected)
        goalie_stats['gsax'] = goalie_stats['xGoals'] - goalie_stats['goals']
        
        # Calculate GSAx per game for normalization
        goalie_stats['gsax_per_game'] = goalie_stats['gsax'] / goalie_stats['games_played']
        
        # Store goalie ratings (keyed by name)
        for _, row in goalie_stats.iterrows():
            self.goalie_ratings[row['name']] = {
                'gsax': row['gsax'],
                'gsax_per_game': row['gsax_per_game'],
                'games': row['games_played'],
                'team': row['team']
            }
    
    def _calculate_weights(self):
        """Calculate blending weights based on games played"""
        g = self.current_games
        
        # Prior season weight (drops to zero after 20 games)
        if g < 10:
            self.prior_w = 0.80  # Heavy reliance early
        elif g < 20:
            self.prior_w = 0.50  # Transition period
        elif g < 30:
            self.prior_w = 0.20  # Minimal reliance
        else:
            self.prior_w = 0.00  # IGNORE prior season after 30 games
        
        current_w = 1 - self.prior_w
        
        # Split current between W/L and metrics
        if g < 10:
            # Very early: mostly metrics
            self.current_wl_w = current_w * 0.20
            self.current_met_w = current_w * 0.80
        elif g < 20:
            # Early: balanced
            self.current_wl_w = current_w * 0.50
            self.current_met_w = current_w * 0.50
        elif g < 40:
            # Mid: favor W/L
            self.current_wl_w = current_w * 0.75
            self.current_met_w = current_w * 0.25
        else:
            # Late: metrics-driven with W/L reality check (after 40 games)
            self.current_wl_w = current_w * 0.40  # 40% W/L
            self.current_met_w = current_w * 0.60  # 60% metrics
    
    def _blend_strengths(self):
        """Blend all strength sources"""
        self.strengths = {}
        
        all_teams = set(self.prior_bt.teams)
        if self.has_bt:
            all_teams.update(self.current_bt.teams)
        all_teams.update(self.current_metrics.metric_strengths.keys())
        
        for team in all_teams:
            # Get each component (default to 1.0 if missing)
            prior = self.prior_bt.team_strengths.get(team, 1.0)
            current_wl = self.current_bt.team_strengths.get(team, 1.0) if self.has_bt else 1.0
            current_met = self.current_metrics.get_team_strength(team)
            
            # Blend
            blended = (self.prior_w * prior + 
                      self.current_wl_w * current_wl + 
                      self.current_met_w * current_met)
            
            self.strengths[team] = blended
        
        # Normalize
        avg = np.mean(list(self.strengths.values()))
        self.strengths = {t: s/avg for t, s in self.strengths.items()}
    
    def get_goalie_adjustment(self, goalie_name):
        """
        Get strength multiplier for a goalie.
        
        Returns: Multiplier (0.90 to 1.10)
        - Elite goalie (top GSAx): 1.08-1.10 boost
        - Average goalie: 1.00 (no change)
        - Bad goalie: 0.90-0.92 penalty
        """
        if not self.goalie_ratings or goalie_name not in self.goalie_ratings:
            return 1.0  # No adjustment if goalie unknown
        
        goalie = self.goalie_ratings[goalie_name]
        gsax_per_game = goalie['gsax_per_game']
        
        # Convert GSAx to multiplier
        # Elite: +0.3 GSAx/game = 1.10x
        # Bad: -0.3 GSAx/game = 0.90x
        multiplier = 1.0 + (gsax_per_game * 0.33)
        
        # Cap between 0.88 and 1.12
        return max(0.88, min(1.12, multiplier))
    
    def predict(self, home_team, away_team, home_advantage=1.08, 
                home_goalie=None, away_goalie=None):
        """
        Predict game outcome.
        
        Parameters:
        - home_team: Home team abbreviation
        - away_team: Away team abbreviation
        - home_advantage: Home ice multiplier (default 8% boost)
        - home_goalie: Starting goalie name (optional)
        - away_goalie: Starting goalie name (optional)
        
        Returns: Dict with probabilities
        """
        home_str = self.strengths[home_team] * home_advantage
        away_str = self.strengths[away_team]
        
        # Apply goalie adjustments
        if home_goalie:
            home_goalie_adj = self.get_goalie_adjustment(home_goalie)
            home_str *= home_goalie_adj
        else:
            home_goalie_adj = 1.0
        
        if away_goalie:
            away_goalie_adj = self.get_goalie_adjustment(away_goalie)
            away_str *= away_goalie_adj
        else:
            away_goalie_adj = 1.0
        
        home_prob = home_str / (home_str + away_str)
        
        return {
            'home_team': home_team,
            'away_team': away_team,
            'home_win_prob': home_prob,
            'away_win_prob': 1 - home_prob,
            'home_goalie_adj': home_goalie_adj,
            'away_goalie_adj': away_goalie_adj
        }
    
    def save(self, filename='nhl_model_data.json'):
        """Save model to file"""
        data = {
            'strengths': self.strengths,
            'current_games': self.current_games,
            'prior_weight': self.prior_w,
            'current_wl_weight': self.current_wl_w,
            'current_metric_weight': self.current_met_w,
            'timestamp': datetime.now().isoformat()
        }
        
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        
        print(f"✅ Model saved to {filename}")

In [7]:
# ============================================================
# PART 3: TRAIN & SAVE MODEL
# ============================================================

# Cell 7: Load Goalie Data & Train Model
query_goalies = """
    SELECT 
        name, team, season, situation, games_played, "xGoals", goals
    FROM "Goalie_stats"
    WHERE season = 2025
"""

goalie_df = pd.read_sql(query_goalies, engine)

# Cast numeric columns
goalie_df[['games_played', 'xGoals', 'goals']] = goalie_df[['games_played', 'xGoals', 'goals']].apply(pd.to_numeric, errors='coerce')

print(f"Loaded {len(goalie_df[goalie_df['situation']=='all'])} goalies\n")

model = NHLPredictionModel(prior_season, current_season, goalie_df)

Loaded 94 goalies

Training model...
  - Prior season Bradley-Terry...
  - Current season: 1082 games
  - Current season Bradley-Terry...
  - Current season metrics...
  - Loading goalie ratings...


AttributeError: 'NHLPredictionModel' object has no attribute 'goalie_ratings'

In [ ]:
# Cell 8: View Top Teams
print("TOP 10 TEAMS:")
strengths_df = pd.DataFrame([
    {'Team': team, 'Strength': strength}
    for team, strength in model.strengths.items()
]).sort_values('Strength', ascending=False)

print(strengths_df.head(10).to_string(index=False))

TOP 10 TEAMS:
Team  Strength
 CAR  4.680901
 COL  3.506217
 OTT  1.681543
 TBL  1.653117
 VGK  1.306531
 PIT  1.053640
 FLA  1.035232
 DAL  1.025383
 UTA  0.940753
 DET  0.921445


In [ ]:

# Cell 9: Test Predictions (with and without goalies)
print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

test_games = [
    ('TOR', 'BOS', 'Joseph Woll', 'Jeremy Swayman'),
    ('VGK', 'COL', 'Adin Hill', 'Alexandar Georgiev'),
    ('WPG', 'EDM', 'Connor Hellebuyck', 'Stuart Skinner')
]

print("\nWithout Goalies:")
for home, away, _, _ in test_games:
    pred = model.predict(home, away)
    print(f"{home} vs {away}: {home} {pred['home_win_prob']:.1%} | {away} {pred['away_win_prob']:.1%}")

print("\nWith Goalies:")
for home, away, home_g, away_g in test_games:
    pred = model.predict(home, away, home_goalie=home_g, away_goalie=away_g)
    print(f"{home} ({home_g}) vs {away} ({away_g}):")
    print(f"  {home} {pred['home_win_prob']:.1%} | {away} {pred['away_win_prob']:.1%}")
    print(f"  Goalie adjustments: {home} {pred['home_goalie_adj']:.2f}x | {away} {pred['away_goalie_adj']:.2f}x")



SAMPLE PREDICTIONS

Without Goalies:
TOR vs BOS: TOR 44.8% | BOS 55.2%
VGK vs COL: VGK 28.7% | COL 71.3%
WPG vs EDM: WPG 38.4% | EDM 61.6%

With Goalies:
TOR (Joseph Woll) vs BOS (Jeremy Swayman):
  TOR 40.7% | BOS 59.3%
  Goalie adjustments: TOR 0.95x | BOS 1.12x
VGK (Adin Hill) vs COL (Alexandar Georgiev):
  VGK 26.2% | COL 73.8%
  Goalie adjustments: VGK 0.88x | COL 1.00x
WPG (Connor Hellebuyck) vs EDM (Stuart Skinner):
  WPG 38.9% | EDM 61.1%
  Goalie adjustments: WPG 1.09x | EDM 1.07x
